In [1]:
import os
from autogen.agentchat import UserProxyAgent, AssistantAgent, GroupChat, GroupChatManager
from autogen.coding import LocalCommandLineCodeExecutor
from dotenv import load_dotenv
from openai import AzureOpenAI
import json
import pandas as pd
import numpy as np
from datetime import datetime
from collections import Counter
load_dotenv()

azure_gpt4o = {
    "api_type": "azure",
    "model": os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'),
    "api_key": os.getenv('OPENAI_API_KEY'),
    "base_url": os.getenv('AZURE_OPENAI_ENDPOINT'),
    "api_version": os.getenv('OPENAI_API_VERSION')
}

flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.


In [2]:
#print(os.getenv('OPENAI_API_VERSION'))
participant_id = 7
transcripts_dir = os.path.join(os.getcwd(), 'transcripts')
data_dir    = os.path.join(transcripts_dir, f"participant{participant_id}")
#blandai_data_dir = os.path.join(os.getcwd(), 'blandai-data')
codebook_file = os.path.join(transcripts_dir, 'codebook.json')
transcripts_file = os.path.join(data_dir, 'blandai_transcripts.json')
#sample_output_file = os.path.join(data_dir, 'example_output.csv')
#print(sample_output_file)
#print(codebook_file)
#print(blandai_data_dir)
#print(transcripts_file)

In [3]:
def get_QA_and_codebook(codebook_file):
    qa_list = []
    with open(codebook_file, 'r') as file:
        codebook = json.load(file)
        question_count = 1
        for code, content in codebook.items():  
            question = content['question']
            qa_list.append(f"Question {question_count}: {question}\nResponse Options: ")
            ans_list = []
            for ans, ans_id in content['clean_response_text_to_id'].items():
                ans_list.append(f"{ans}")
            #if code in ['HH01S', 'HH25S', 'HH612S', 'HH1317S', 'HH18OVS', 'PHYS11_TEMP']:
            #    ans_list = ['Numeric Value']
                
            qa_list.append('; '.join(ans_list))
            qa_list.append("\n")
            question_count += 1
    return ''.join(qa_list), codebook
QA_details, codebook = get_QA_and_codebook(codebook_file)
question_count = len(codebook)
print(QA_details)
print(question_count)

Question 1: What is your current age?
Response Options: 18-24; 25-34; 35-44; 45-54; 55-64; 65-74; 75+; Under 18
Question 2: Are you male or female?
Response Options: Male; Female; REFUSED
Question 3: What race or races you consider yourself to be? You can say multiple races.
Response Options: White; Black or African American; American Indian or Alaska Native; Asian Indian; Chinese; Filipino; Japanese; Korean; Vietnamese; Other Asian; Native Hawaiian; Guamanian or Chamorro; Samoan; Other Pacific Islander; Some other race; REFUSED
Question 4: What was your total HOUSEHOLD income in 2019?
Response Options: Under $10,000; $10,000 to under $20,000; $20,000 to under $30,000; $30,000 to under $40,000; $40,000 to under $50,000; $50,000 to under $75,000; $75,000 to under $100,000; $100,000 to under $150,000; $150,000 or more; DON'T KNOW; REFUSED
Question 5: What is the highest level of school you have completed?
Response Options: No formal education; 1st, 2nd, 3rd, or 4th grade; 5th or 6th grad

In [4]:
client = AzureOpenAI(
  api_key = os.getenv('OPENAI_API_KEY'),  
  api_version = os.getenv('OPENAI_API_VERSION'),
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
)



role_description = f''' You are a helpful assistant that reads survey conversation transcript and deduces responses given by user to each question.
                        There could have been errors while transcribing conversations.
                        You return an output with the responses for each question in order of they appear in the question list.
                        The returned output needs to be a SINGLE LINE, with individual responses separated by semicolons and no other punctuation.
                        Format of the output line should be: Question 1: Response 1; Question 2: Response 2; etc. 
                        Each deduced response for a question should be STRICTLY selected from corresponding Reponse Options. 
                        If Reponse Options includes 'Numeric Value' as an option, deduce actual numeric value from conversation.
                        Each question should have an answer, if question has Numeric Value as an option you couldn't deduce the response put 'NaN'.
                        If you are absolutely sure that there is no matching response option for a question and participant didn't explicitly refuse to answer then put 'NA'.
                        Make sure that number of answers equals number of questions.
                        
                        Below is the list of questions, each followed on the next line by its corresponding response options separated by semicolons:
                        {QA_details}
                        '''
#print(role_description)



with open(transcripts_file, 'r') as file:
    transcripts = json.load(file)



def generate_response(conversation, question_count, attempts_per_transcript):
    question_to_reponses = {i:[] for i in range(question_count)} 
    for _ in range(attempts_per_transcript):
        response = client.chat.completions.create(
            model=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'), # model = "deployment_name".
            messages=conversation
        )
        responses_in_one_line = response.choices[0].message.content
        user_responses = [part.strip() for part in responses_in_one_line.split(';')]
        while len(user_responses) != question_count:
            print(f"Number of responses does not match expected number for {user_id}, got: {len(user_responses)}, expected: {question_count}")
            response = client.chat.completions.create(
                model=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'), # model = "deployment_name".
                messages=conversation
            )
            user_responses = [part.strip() for part in userid_to_answers[user_id].split(';')]
        for i in range(question_count):
            response = user_responses[i]
            answer   = response[response.index(':')+1:].strip()
            answer   = 'nan' if answer == 'NaN' else answer
            question_to_reponses[i].append(answer)

    print("Here are all of the answers:")
    print(question_to_reponses)
    finalized_responses = []
    for i in range(question_count):
        counter = Counter(question_to_reponses[i])
        most_frequent = counter.most_common(1)[0][0]
        finalized_responses.append(most_frequent)

    print("Final response selections: ")
    print(finalized_responses)
    return finalized_responses
        
        

task_prompt = f"""Help me to understand the following conversation transcript : 

{transcripts["0"]}"""
#print(task_prompt)

conversation=[{"role": "system", "content": role_description}]
userid_to_answers = {}

attempts_per_transcript = 5

for user_id, transcript in transcripts.items():
    survey = transcript.replace('user:', 'respondent:')
    survey = survey.replace('assistant:', 'surveyor:')
    task_prompt = f"""Help me to understand the following conversation transcript : 
                    {survey}"""
    conversation.append({"role": "user", "content": task_prompt})

    #print(task_prompt)
    #print(survey)
    '''
    response = client.chat.completions.create(
        model=os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME'), # model = "deployment_name".
        messages=conversation
    )
    responses_in_one_line = response.choices[0].message.content    
    userid_to_answers[user_id] = responses_in_one_line
    print(userid_to_answers[user_id])'''

    person_responses = generate_response(conversation, question_count, attempts_per_transcript)
    userid_to_answers[user_id] = person_responses
    conversation.pop()

Here are all of the answers:
{0: ['35-44', '35-44', '35-44', '35-44', '35-44'], 1: ['Male', 'Male', 'Male', 'Male', 'Male'], 2: ['Asian Indian', 'Asian Indian', 'Asian Indian', 'Asian Indian', 'Asian Indian'], 3: ['$40,000 to under $50,000', '$40,000 to under $50,000', '$40,000 to under $50,000', '$40,000 to under $50,000', '$40,000 to under $50,000'], 4: ['High school graduate - high school diploma or the equivalent', 'High school graduate - high school diploma or the equivalent', 'High school graduate - high school diploma or the equivalent', 'High school graduate - high school diploma or the equivalent', 'High school graduate - high school diploma or the equivalent'], 5: ['Six or more persons', 'Six or more persons', 'Six or more persons', 'Six or more persons', 'Six or more persons'], 6: ['1', '1', '1', '1', '1'], 7: ['2', '2', '2', '2', '2'], 8: ['2', '2', '2', '2', '2'], 9: ['0', '0', '0', '0', '0'], 10: ['2', '2', '2', '2', '2'], 11: ['A few times a month', 'A few times a month'

In [5]:
answers_as_list = []
'''
for user_id in sorted(list(userid_to_answers.keys())):
    user_responses = [part.strip() for part in userid_to_answers[user_id].split(';')]
    clean_user_responses = []
    for response in user_responses:
        clean_text = response[response.index(':')+1:].strip()
        if clean_text == 'NaN':
            clean_user_responses.append('nan')
        else:
            clean_user_responses.append(clean_text)
    answers_as_list.append(clean_user_responses)'''

for user_id in sorted(list(userid_to_answers.keys())):
    
    answers_as_list.append(userid_to_answers[user_id])
    print(len(userid_to_answers[user_id]))
    print(userid_to_answers[user_id])

33
['35-44', 'Male', 'Asian Indian', '$40,000 to under $50,000', 'High school graduate - high school diploma or the equivalent', 'Six or more persons', '1', '2', '2', '0', '2', 'A few times a month', 'A few times a week', '1-2 days', 'Not at all or less than 1 day', '5-7 days', '1-2 days', 'Not at all or less than 1 day', "Yes, I worked for someone else for wages, salary, piece rate, commission, tips, or payments 'in kind,' for example, food or lodging received as payment for work performed", 'Good', 'No', 'No', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes', 'Yes', '100.4']
33
['25-34', 'Female', 'REFUSED', '$75,000 to under $100,000', '7th or 8th grade', 'Four persons', '0', '0', '0', '0', '4', 'A few times a week', 'A few times a week', 'Not at all or less than 1 day', '1-2 days', '1-2 days', '1-2 days', 'Not at all or less than 1 day', "Yes, I worked for someone else for wages, salary, piece rate, commission, tips, or payments 'in kind,' for example, food or lodging receive

In [6]:
question_code_to_text_responses = {}
i = 0
#print(codebook['SOC1']['answer_to_answer_id']['Some'])
for code, val in codebook.items():
    print(code)
    question_code_to_text_responses[code] = []
    for user_answers in answers_as_list:
        response_text = user_answers[i]
        question_code_to_text_responses[code].append(response_text)
    i += 1

df = pd.DataFrame(question_code_to_text_responses)
df.head()

AGE7
GENDER
RACETH
HHINCOME
EDUCATION
HHSIZE1
HH01S
HH25S
HH612S
HH1317S
HH18OVS
SOC2A
SOC2B
SOC5A
SOC5B
SOC5C
SOC5D
SOC5E
ECON1
PHYS8
PHYS4
PHYS5
PHYS1B
PHYS1C
PHYS1D
PHYS1E
PHYS1F
PHYS1G
PHYS1H
PHYS1I
PHYS1J
PHYS11
PHYS11_TEMP


,AGE7,GENDER,RACETH,HHINCOME,EDUCATION,HHSIZE1,HH01S,HH25S,HH612S,HH1317S,...,PHYS1C,PHYS1D,PHYS1E,PHYS1F,PHYS1G,PHYS1H,PHYS1I,PHYS1J,PHYS11,PHYS11_TEMP
0,35-44,Male,Asian Indian,"$40,000 to under $50,000",High school graduate - high school diploma or ...,Six or more persons,1,2,2,0,...,Yes,No,No,No,No,No,No,Yes,Yes,100.4
1,25-34,Female,REFUSED,"$75,000 to under $100,000",7th or 8th grade,Four persons,0,0,0,0,...,No,No,No,No,No,No,No,No,No,nan
2,18-24,Male,Black or African American,"$20,000 to under $30,000",7th or 8th grade,Six or more persons,1,2,2,0,...,Yes,Yes,No,No,No,No,No,No,No,nan
3,65-74,Female,Chinese,REFUSED,High school graduate - high school diploma or ...,"One person, I live by myself",0,0,0,0,...,Yes,No,No,No,No,Yes,No,No,Yes,97.7
4,55-64,Female,Asian Indian,"$150,000 or more",Professional or Doctorate degree,"One person, I live by myself",0,0,0,0,...,No,No,No,Yes,No,No,No,No,Yes,97.6


In [7]:
gpt_res_dir = os.path.join(os.getcwd(),  'gpt-deductions')
if not os.path.exists(gpt_res_dir):
    os.makedirs(gpt_res_dir)

In [8]:
# Save conversation transcripts
fname  = os.path.join(gpt_res_dir,  f"participant{participant_id}_deduced_{datetime.today().strftime('%Y-%m-%d')}.csv")
df.to_csv(fname, index=False)
df = pd.read_csv(fname)
#df = df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0', 'Unnamed: 0.2'])
df.head()

,AGE7,GENDER,RACETH,HHINCOME,EDUCATION,HHSIZE1,HH01S,HH25S,HH612S,HH1317S,...,PHYS1C,PHYS1D,PHYS1E,PHYS1F,PHYS1G,PHYS1H,PHYS1I,PHYS1J,PHYS11,PHYS11_TEMP
0,35-44,Male,Asian Indian,"$40,000 to under $50,000",High school graduate - high school diploma or ...,Six or more persons,1,2,2,0,...,Yes,No,No,No,No,No,No,Yes,Yes,100.4
1,25-34,Female,REFUSED,"$75,000 to under $100,000",7th or 8th grade,Four persons,0,0,0,0,...,No,No,No,No,No,No,No,No,No,NaN
2,18-24,Male,Black or African American,"$20,000 to under $30,000",7th or 8th grade,Six or more persons,1,2,2,0,...,Yes,Yes,No,No,No,No,No,No,No,NaN
3,65-74,Female,Chinese,REFUSED,High school graduate - high school diploma or ...,"One person, I live by myself",0,0,0,0,...,Yes,No,No,No,No,Yes,No,No,Yes,97.7
4,55-64,Female,Asian Indian,"$150,000 or more",Professional or Doctorate degree,"One person, I live by myself",0,0,0,0,...,No,No,No,Yes,No,No,No,No,Yes,97.6
